# Convolutions for Images
:label:`sec_conv_layer`

- Now that we understand how **convolutional layers work in theory**,
  - We are ready to see **how they work in practice**.

- We continue with **images as our running example**:
  - Convolutional neural networks are **efficient architectures** for capturing the **structure** in image data.


In [1]:
import torch
from torch import nn
from d2l import torch as d2l

## The Cross-Correlation Operation

- Strictly speaking, **convolutional layers** are somewhat of a **misnomer**:
  - The operation they perform is more accurately described as a **cross-correlation**.

- As described in :numref:`sec_why-conv`:
  - A convolutional layer combines an **input tensor** and a **kernel tensor**
  - Through a **cross-correlation operation** to produce an **output tensor**.

- To illustrate the concept, we **ignore channels** and consider:
  - **Two-dimensional input** and **hidden representations**.

- In :numref:`fig_correlation`:
  - The input is a 2D tensor with **height 3** and **width 3**:
    - Shape is denoted as $3 \times 3$ or $(3, 3)$.
  - The kernel has **height 2** and **width 2**:
    - So the **kernel window** is $2 \times 2$.

![Two-dimensional cross-correlation operation. The shaded portions are the first output element as well as the input and kernel tensor elements used for the output computation: $0\times0+1\times1+3\times2+4\times3=19$.](../img/correlation.svg)
:label:`fig_correlation`


- In the **two-dimensional cross-correlation operation**:
  - We start with the **convolution window** at the **upper-left corner** of the input tensor.
  - We **slide** the window **from left to right** and **top to bottom** across the input.

- At each position:
  - The input **subtensor** within the window is **multiplied elementwise** with the kernel.
  - The **resulting values are summed**, yielding a **single scalar**.
  - This scalar is placed in the **output tensor** at the corresponding location.

- In the current example:
  - The **output tensor** has shape $2 \times 2$.
  - Its four elements result from the following computations:

  $$
  \begin{aligned}
  0\times0+1\times1+3\times2+4\times3 &= 19,\\
  1\times0+2\times1+4\times2+5\times3 &= 25,\\
  3\times0+4\times1+6\times2+7\times3 &= 37,\\
  4\times0+5\times1+7\times2+8\times3 &= 43.
  \end{aligned}
  $$

- Note that the **output size is smaller** than the input size:
  - Since the kernel must **fit entirely within the image** to compute a valid output.
  - If the input size is $n_\textrm{h} \times n_\textrm{w}$ and kernel size is $k_\textrm{h} \times k_\textrm{w}$,
    - Then the output size is:

    $$
    (n_\textrm{h}-k_\textrm{h}+1) \times (n_\textrm{w}-k_\textrm{w}+1).
    $$

- This output shape arises because:
  - We need enough room to **shift the kernel** fully across the image.

- Later, we will see how to **keep the output size unchanged**:
  - By **padding the input** with zeros around its border.

- Next, we implement this process in a function `corr2d`:
  - It accepts:
    - An **input tensor** `X`.
    - A **kernel tensor** `K`.
  - It returns an **output tensor** `Y`.


In [2]:
def corr2d(X, K):  #@save
    """Compute 2D cross-correlation."""
    h, w = K.shape
    Y = torch.zeros((X.shape[0] - h + 1, X.shape[1] - w + 1))
    for i in range(Y.shape[0]):
        for j in range(Y.shape[1]):
            Y[i, j] = (X[i:i + h, j:j + w] * K).sum()
    return Y

- We can construct the **input tensor** `X` and the **kernel tensor** `K` from :numref:`fig_correlation`.

- This allows us to **validate the output** of the above implementation of the **two-dimensional cross-correlation operation**.


In [3]:
X = torch.tensor([[0.0, 1.0, 2.0], [3.0, 4.0, 5.0], [6.0, 7.0, 8.0]])
K = torch.tensor([[0.0, 1.0], [2.0, 3.0]])
corr2d(X, K)

tensor([[19., 25.],
        [37., 43.]])

## Convolutional Layers

- A **convolutional layer**:
  - Performs **cross-correlation** between the input and the kernel.
  - **Adds a scalar bias** to produce the output.

- The two **parameters** of a convolutional layer are:
  - The **kernel**.
  - The **scalar bias**.

- During training:
  - The **kernel is typically initialized randomly**, similar to fully connected layers.

- We are now ready to **implement a two-dimensional convolutional layer**:
  - Based on the `corr2d` function defined earlier.

- In the `__init__` method:
  - We declare `weight` and `bias` as the two **learnable parameters**.

- In the **forward propagation** method:
  - We call `corr2d` on the input and kernel.
  - Then we **add the bias** to the result.


In [4]:
class Conv2D(nn.Module):
    def __init__(self, kernel_size):
        super().__init__()
        self.weight = nn.Parameter(torch.rand(kernel_size))
        self.bias = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        return corr2d(x, self.weight) + self.bias

- In an $h \times w$ **convolution** or an $h \times w$ **convolution kernel**:
  - The **height** of the kernel is $h$.
  - The **width** of the kernel is $w$.

- A convolutional layer using an $h \times w$ kernel is referred to as an **$h \times w$ convolutional layer**.

## Object Edge Detection in Images

- Let’s examine a simple application of a **convolutional layer**:
  - **Detecting the edge of an object** in an image.
  - This involves **finding the location** where pixel values **change**.

- First, we construct a synthetic **image of size $6 \times 8$ pixels**:
  - The **middle four columns** are black (`0`).
  - The **remaining columns** are white (`1`).


In [5]:
X = torch.ones((6, 8))
X[:, 2:6] = 0
X

tensor([[1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.]])

- Next, we construct a **kernel `K`** with:
  - Height of 1
  - Width of 2

- When performing the **cross-correlation** with the input:
  - If **horizontally adjacent elements are equal**, the output is **0**.
  - If they are **different**, the output is **nonzero**.

- This kernel is a **special case of a finite difference operator**:
  - At location $(i, j)$, it computes:
    $$
    x_{i,j} - x_{(i+1),j}
    $$
  - This represents the **difference between horizontally adjacent pixels**.

- It is a **discrete approximation** of the **first derivative** in the **horizontal direction**.

- For a function $f(i, j)$, the derivative is:
  $$
  -\partial_i f(i,j) = \lim_{\epsilon \to 0} \frac{f(i,j) - f(i+\epsilon,j)}{\epsilon}
  $$

- Let’s see how this behaves **in practice**.


In [6]:
K = torch.tensor([[1.0, -1.0]])

- We now perform the **cross-correlation operation** using:
  - `X` as the **input tensor**
  - `K` as the **kernel**

- As a result:
  - We detect **`1`** at the **edge from white to black**
  - We detect **`-1`** at the **edge from black to white**
  - All other positions yield an output of $0$


In [7]:
Y = corr2d(X, K)
Y

tensor([[ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.]])

- We can now apply the kernel to the transposed image.
- As expected, it vanishes. [**The kernel `K` only detects vertical edges.**]


In [8]:
corr2d(X.t(), K)

tensor([[0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.]])

## Learning a Kernel

- Designing an edge detector using **finite differences** like `[1, -1]` is convenient,
  - But only if we **already know** what we are looking for.

- As we consider:
  - **Larger kernels**, and
  - **Multiple layers** of convolutions,
  - It becomes **impractical** to manually specify what each filter should do.

- Now let's see whether we can **learn the kernel that generated `Y` from `X`**:
  - Using only **input--output pairs**.

- We proceed as follows:
  - Construct a **convolutional layer**.
  - Initialize its kernel with a **random tensor**.
  - In each iteration:
    - Use **squared error** to compare the layer’s output to `Y`.
    - **Compute gradients** to update the kernel.

- For simplicity:
  - We use the **built-in class** for 2D convolutional layers.
  - We **ignore the bias** in this example.


In [9]:
# Construct a two-dimensional convolutional layer with 1 output channel and a
# kernel of shape (1, 2). For the sake of simplicity, we ignore the bias here
conv2d = nn.LazyConv2d(1, kernel_size=(1, 2), bias=False)

# The two-dimensional convolutional layer uses four-dimensional input and
# output in the format of (example, channel, height, width), where the batch
# size (number of examples in the batch) and the number of channels are both 1
X = X.reshape((1, 1, 6, 8))
Y = Y.reshape((1, 1, 6, 7))
lr = 3e-2  # Learning rate

for i in range(10):
    Y_hat = conv2d(X)
    l = (Y_hat - Y) ** 2
    conv2d.zero_grad()
    l.sum().backward()
    # Update the kernel
    conv2d.weight.data[:] -= lr * conv2d.weight.grad
    if (i + 1) % 2 == 0:
        print(f'epoch {i + 1}, loss {l.sum():.3f}')

epoch 2, loss 16.481
epoch 4, loss 5.069
epoch 6, loss 1.794
epoch 8, loss 0.688
epoch 10, loss 0.274


- Note that the error has dropped to a small value after 10 iterations.
- Now we will [**take a look at the kernel tensor we learned.**]


In [10]:
conv2d.weight.data.reshape((1, 2))

tensor([[ 1.0398, -0.9328]])

- The **learned kernel tensor** turns out to be **remarkably close** to the original kernel tensor `K` we defined earlier.

## Cross-Correlation and Convolution

- As noted in :numref:`sec_why-conv`, there is a correspondence between:
  - The **cross-correlation** and **convolution** operations.

- We continue with the case of **two-dimensional convolutional layers**.

- What if the layer performs **strict convolution** (as in :eqref:`eq_2d-conv-discrete`) instead of cross-correlation?
  - To get the same result:
    - **Flip the kernel tensor** both **horizontally** and **vertically**.
    - Then apply the **cross-correlation** operation.

- Important point:
  - Since kernels are **learned from data**, the output remains the same regardless of:
    - Whether the layer performs **strict convolution** or **cross-correlation**.

- Illustration:
  - Suppose a layer performs **cross-correlation** and learns kernel $\mathbf{K}$ (as shown in :numref:`fig_correlation`).
  - If the same layer instead performs **strict convolution**, it will learn a kernel $\mathbf{K}'$ such that:
    - $\mathbf{K}'$ is obtained by **flipping** $\mathbf{K}$ both horizontally and vertically.
    - The output remains **identical** to the result in :numref:`fig_correlation`.

- In deep learning literature:
  - It is standard to **refer to cross-correlation as convolution**.
  - Even though there is a **slight technical difference**.
  - We also use the term **element** to refer to:
    - An **entry** or **component** of a tensor (either in a layer representation or a convolution kernel).


## Feature Map and Receptive Field

- As discussed in :numref:`subsec_why-conv-channels`, the output of a convolutional layer (e.g., in :numref:`fig_correlation`) is called a **feature map**:
  - It represents **learned features** in the **spatial dimensions** (e.g., width and height).
  - These features are passed to the **next layer** in the network.

- In CNNs, for any element $x$ in a layer, its **receptive field** refers to:
  - All elements from previous layers that may **influence the calculation** of $x$ during forward propagation.
  - The receptive field can be **larger than the input size**.

- Example using :numref:`fig_correlation`:
  - With a $2 \times 2$ convolution kernel, the **receptive field** of the output element (value $19$) includes the **four shaded input elements**.
  - If we apply a second $2 \times 2$ convolutional layer on the $2 \times 2$ output (denoted $\mathbf{Y}$), producing a single output $z$:
    - The **receptive field of $z$ on $\mathbf{Y}$** includes all four elements of $\mathbf{Y}$.
    - The **receptive field on the original input** now includes **all nine input elements**.

- This shows that:
  - **Deeper networks** naturally lead to **larger receptive fields**.
  - This helps detect **broader patterns** in the input.

- The term **receptive field** originates from **neurophysiology**:
  - Experiments :cite:`Hubel.Wiesel.1959,Hubel.Wiesel.1962,Hubel.Wiesel.1968` studied how the **visual cortex** responds to various stimuli.
  - They found that **lower levels** in the visual hierarchy respond to **edges and simple shapes**.
  - Later, :citet:`Field.1987` showed similar results using **natural images and convolution-like filters**.

![Figure and caption taken from :citet:`Field.1987`: An example of coding with six different channels. (Left) Examples of the six types of sensor associated with each channel. (Right) Convolution of the image in (Middle) with the six sensors shown in (Left). The response of the individual sensors is determined by sampling these filtered images at a distance proportional to the size of the sensor (shown with dots). This diagram shows the response of only the even symmetric sensors.](../img/field-visual.png)
:label:`field_visual`

- Interestingly, this relationship also holds for **deeper layers** in modern CNNs:
  - As shown in :citet:`Kuzovkin.Vicente.Petton.ea.2018`, deeper layers compute features that resemble those found in **biological systems**.

- In summary:
  - **Convolutions are a powerful tool** in computer vision.
  - Their effectiveness is evident in **both biology and deep learning systems**.
  - It is not surprising, in hindsight, that **convolutional networks were central** to the success of deep learning.


## Summary

- The core computation of a **convolutional layer** is the **cross-correlation operation**.
- A **simple nested for-loop** is sufficient to compute it.
- With **multiple input and output channels**, this becomes a **matrix--matrix operation across channels**.

- The computation is:
  - **Straightforward**
  - Most importantly, **highly local**

- **Hardware advantages**:
  - Local computation enables **significant hardware optimization**.
  - Many computer vision advances are made possible by the ability to optimize convolutions efficiently.
  - Chip designers can focus on **fast computation** rather than **large memory**.

- **Applications of convolutions**:
  - Detecting **edges** and **lines**
  - **Blurring** or **sharpening** images

- **Key insight**:
  - We do **not** need to manually engineer filters.
  - Filters can be **learned from data**, replacing heuristics with **evidence-based statistics**.

- **Biological relevance**:
  - Learned filters resemble **receptive fields** and **feature maps** in the brain.
  - This provides **confidence** in the use of convolutions for building deep networks.
